# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
This dataset follows the Croissant metadata standard and can be accessed via the schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant --quiet

## 1. Data Loading

Load dataset metadata and records via `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the mlcroissant Dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview

List available record sets, their `@id`s, and the fields within them. This helps identify the data structure for further exploration.

> We'll use the `dataset.record_sets` attribute, which provides RecordSet objects. Each RecordSet exposes its `@id` and fields.

In [ ]:
# Examine available record sets and their fields (all by @id)
all_record_sets = list(dataset.record_sets)
print(f"Number of record sets found: {len(all_record_sets)}\n")
for rs in all_record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} (name: {field.name})")
    print()

## 3. Data Extraction

Extract all record sets as pandas DataFrames by their `@id`. This facilitates analysis of multiple tables. Use the `@id` values shown above, keeping all processing dynamic.

> All entities—record sets, fields, columns—are referenced **by their `@id`** (as required).

In [ ]:
# Get the list of all record set @id's
record_set_ids = [rs.id for rs in dataset.record_sets]
print("Loaded record set @ids:")
for rid in record_set_ids:
    print(f"  {rid}")

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Display first record set's columns to use below
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head(3))
else:
    print("No record sets found in this dataset.")

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field and a grouping field (by their `@id`), perform filtering, normalization, and optionally grouping, **all referencing by `@id`** as required.

*Please update the variable assignments below to select different fields for processing from the DataFrame's columns.*

In [ ]:
# Replace the below with the desired record set @id. Use the displayed record sets and fields above.

# -- Example: dynamically choose the first record set and logical numeric/categorical fields for demo.
if record_set_ids:
    rs_id = first_rs_id
    df = dataframes[rs_id]
    
    print(f"Working with record set @id: {rs_id}")
    # List some candidate numeric and category fields:
    print("Available columns:")
    print(df.columns.tolist())
    
    # Try to select the first float or integer-appearing field for numeric demo
    numeric_field = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field = c
            break
    if numeric_field is None:
        # Fallback: choose any field and try coercion
        numeric_field = df.columns[0]
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    print(f"Selected numeric field (by @id): {numeric_field}")
    
    # Try to select a non-numeric/categorical/groupable field
    group_field = None
    for c in df.columns:
        if c != numeric_field and df[c].nunique() > 1 and df[c].nunique() < len(df) // 2:
            group_field = c
            break
    if not group_field:
        group_field = df.columns[1] if len(df.columns) > 1 else numeric_field
    print(f"Selected group field (by @id): {group_field}\n")

    # Remove outliers and filter
    threshold = df[numeric_field].quantile(0.75)  # Use 75th percentile as example threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping example (mean aggregation)
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No record sets available for EDA.")

## 5. Visualization

Visualize distributions of filtered and normalized data, by numeric and group fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and not filtered_df.empty:
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    sns.histplot(filtered_df[numeric_field], bins=15, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")

    plt.subplot(1,2,2)
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=15, kde=True, color='orange')
    plt.title(f"Normalized {numeric_field}")
    plt.tight_layout()
    plt.show()

    # If group_field is categorical, show boxplot by group
    if group_field in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()


## 6. Conclusion

This notebook demonstrated how to:

1. Load and explore metadata from a Croissant-compliant dataset using `mlcroissant`.
2. Inspect record sets, fields, and their `@id`s.
3. Extract data dynamically into DataFrames based on record set `@id`s.
4. Perform basic EDA—filtering, normalization, and grouping—using only `@id` references for all data structures.
5. Visualize numeric data distributions and field groupings.

**Next Steps**: Customize field and record set choices in the above code per your domain questions, and use the notebook as a template for other Croissant datasets. For reproducibility, always reference each entity by its `@id` as shown.